In [0]:
CREATE OR REFRESH MATERIALIZED VIEW bridge_monitoring.`02_silver`.bridge_metadata
    COMMENT "Static metadata for five major European bridges"
    AS
    SELECT * FROM VALUES
        (1, 'Millau Viaduct',     'France',    2460,  342, 343, 'Tarn Valley', 'Cable-statyed viaduct', 2004),
        (2, 'Tower Bridge',       'UK',        1748,  141, 42.5, 'London', 'Suspension bridge',     1894),
        (3, 'Vasco Da Gama Bridge', 'Portual',       17280, 420, 155, 'Lisbon',   'Suspension bridge',     1937),
        (4, 'Chunnel',            'UK/France',   50,   50, -75, 'Calais',       'Tunnel',                1994),
        (5, 'Seine',              'France',     732,  732, 7, 'Paris',        'River',                 1831)
    AS bridges(bridge_id, name, country, length_m, main_span_m, height_m, location, type, opened_year);

In [0]:
CREATE OR REFRESH STREAMING TABLE bridge_monitoring.`02_silver`.bridge_temperature
    (
        CONSTRAINT valid_event_time EXPECT(event_time IS NOT NULL) ON VIOLATION DROP ROW,
        CONSTRAINT valid_temperature_range EXPECT(temperature BETWEEN -20 AND 60)
    )
    COMMENT "Temperature readings for five major European bridges"
    AS
    SELECT 
        b.bridge_id AS bridge_id,
        CAST(t.event_time AS TIMESTAMP) AS event_time,
        b.name AS bridge_name,
        b.location,
        b.country,
        t.temperature AS temperature
    FROM STREAM(bridge_monitoring.`01_bronze`.bridge_temperature) t
    LEFT JOIN bridge_monitoring.02_silver.bridge_metadata b
    ON t.device_id = b.bridge_id

    

In [0]:
CREATE OR REFRESH STREAMING TABLE bridge_monitoring.`02_silver`.bridge_vibration
    (
        CONSTRAINT valid_event_time EXPECT (event_time IS NOT NULL) ON VIOLATION DROP ROW,
        CONSTRAINT valid_vibration_range EXPECT (vibration BETWEEN 0.0 AND 0.1)
    )
    COMMENT "Vibration readings for five major European bridges"
    AS
    SELECT 
        b.bridge_id AS bridge_id,
        CAST(v.event_time AS TIMESTAMP) AS event_time,
        b.name AS bridge_name,
        b.location,
        b.country,
        v.vibration AS vibration
    FROM STREAM(bridge_monitoring.`01_bronze`.bridge_vibration) v
    JOIN bridge_monitoring.02_silver.bridge_metadata b
    ON v.device_id = b.bridge_id

In [0]:
CREATE OR REFRESH STREAMING TABLE bridge_monitoring.`02_silver`.bridge_tilt
    (
        CONSTRAINT valid_bridge_id EXPECT (bridge_id IS NOT NULL) ON VIOLATION DROP ROW,
        CONSTRAINT valid_tilt_range EXPECT (tilt_angle BETWEEN -0.005 AND 0.005)
    )
    COMMENT "Tilt readings for five major European bridges"
    AS
    SELECT 
        b.bridge_id AS bridge_id,
        CAST(t.event_time AS TIMESTAMP) AS event_time,
        b.name AS bridge_name,
        b.location,
        b.country,
        t.tilt AS tilt_angle
    FROM STREAM(bridge_monitoring.`01_bronze`.bridge_tilt) t
    JOIN bridge_monitoring.02_silver.bridge_metadata b
    ON t.device_id = b.bridge_id

In [0]:
-- %python
-- import dlt
-- from pyspark.sql.functions import col

-- @dlt.table(name="02_silver.bridge_temperature", comment="Temperature data for five major European bridges")
-- @dlt.expect_or_drop("valid_event_time", "event_time IS NOT NULL")
-- @dlt.expect("valid_temperature_range", "temperature BETWEEN -20 AND 60")
-- def silver_bridge_temperature():
--     return (
--         dlt.read_stream("01_bronze.bridge_temperature")
--         .withColumn("event_time", col("event_time").cast("timestamp"))
--         .join(dlt.read("02_silver.bridge_metadata"), on="bridge_id", how="left")
--         .select(
--             col("bridge_id"), col("name").alias("bridge_name"), col("location"),
--             col("country"), col("event_time"), col("temperature")
        -- ))
-- # Error fixed by importing dlt

In [0]:
-- SELECT * FROM bridge_monitoring.`02_silver`.bridge_tilt

In [0]:
-- SELECT * FROM bridge_monitoring.`02_silver`.bridge_temperature

In [0]:
-- SELECT * FROM bridge_monitoring.`02_silver`.bridge_vibration